[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_01/02_gauss_stokes_campos_y_fuentes.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 2 — Gauss, Stokes, campos y fuentes

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 1**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Usar la ley de Gauss para obtener $\mathbf{D}$ dentro y fuera de una
   esfera cargada.
2. Usar la ley de Ampère para obtener $\mathbf{H}$ alrededor de un conductor
   con corriente.
3. Relacionar corriente, conductividad y campo con la ley de Ohm.
4. Calcular la fuerza de Lorentz sobre una carga en movimiento y determinar
   su dirección.

In [ ]:
# Preparación del entorno: local o Google Colab, con verificación SHA256.
import hashlib
import sys
import urllib.request
from pathlib import Path

MODULOS = {
    "utilidades_notebook.py": "e1811892d086ca99e03694c0e70853886f63be58326e3e7232c6978db1fdf7a9",
    "constantes_fisicas.py": "394c39ad2aac2f4870620e3df6e276046f04abb811d1b28b6fa14f1836fe20ce",
    "campos_electrostaticos.py": "d7ccfcce6a4cb2547f71f1f05bdb8f3912589035677b46ca8547a36cee0d072d",
}
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


candidatos = [Path.cwd(), *Path.cwd().parents]
raiz_repo = next((p for p in candidatos if (p / ".git").exists()), None)
if raiz_repo is not None:
    for modulo, esperado in MODULOS.items():
        archivo = raiz_repo / "src" / modulo
        if not archivo.exists() or sha256(archivo) != esperado:
            raise RuntimeError(
                f"Hash local desactualizado para {modulo}. "
                "Ejecute scripts/refresh_notebook_hashes.py."
            )
    raiz = raiz_repo
else:
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo, esperado in MODULOS.items():
        destino = raiz / "src" / modulo
        if destino.exists() and sha256(destino) == esperado:
            continue
        with urllib.request.urlopen(URL_SRC + modulo, timeout=30) as respuesta:
            datos = respuesta.read()
        obtenido = hashlib.sha256(datos).hexdigest()
        if obtenido != esperado:
            raise RuntimeError(
                f"SHA256 inválido para {modulo}: {obtenido} != {esperado}"
            )
        destino.write_bytes(datos)

sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from constantes_fisicas import EPSILON_0, CARGA_ELEMENTAL
from campos_electrostaticos import (
    carga_total_esfera_uniforme,
    densidad_flujo_esfera_uniforme,
    campo_esfera_uniforme,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 La ley de Gauss sola no basta

La ley de Gauss siempre es cierta, pero no siempre sirve para calcular. Dice
cuánto flujo total sale de una superficie cerrada, no cómo se reparte.

Para despejar $\mathbf{D}$ hace falta una **simetría** que garantice que el
campo vale lo mismo en toda la superficie elegida. Entonces sale de la
integral y el problema se resuelve en dos líneas.

En una esfera con carga uniforme, la simetría esférica garantiza que
$\mathbf{D}$ apunta radialmente y solo depende de $r$. Eligiendo una esfera
concéntrica, el flujo es $D(r)\,4\pi r^2$ y listo.

### 2.2 Dos regiones, dos comportamientos

Dentro de la esfera solo cuenta la carga encerrada hasta el radio $r$, que
crece como $r^3$. Al dividir por el área $4\pi r^2$, queda un campo que crece
**linealmente** con $r$.

Fuera ya no hay carga nueva que sumar: el campo cae como $1/r^2$, igual que
el de una carga puntual.

### 2.3 Ohm y Lorentz

La ley de Ohm $\mathbf{J} = \sigma\mathbf{E}$ no es una ley fundamental: es
una **relación constitutiva** que describe cómo responde el material. En un
buen conductor $\sigma$ es enorme, así que basta un campo diminuto para
sostener una corriente apreciable.

La fuerza de Lorentz es lo que conecta los campos con el movimiento. En el
fondo es la definición operacional de $\mathbf{E}$ y $\mathbf{B}$: son lo que
empuja a las cargas.

## 3. Ecuaciones

**Ley de Gauss:**

$$
\oint_S \mathbf{D}\cdot d\mathbf{s} = Q_{\text{enc}},
\qquad
\nabla\cdot\mathbf{D} = \rho_v .
$$

**Esfera de radio $a$ con $\rho_v$ uniforme:**

$$
Q = \rho_v \frac{4}{3}\pi a^3,
\qquad
D(r) =
\begin{cases}
\dfrac{\rho_v r}{3}, & r < a, \\[2ex]
\dfrac{Q}{4\pi r^2}, & r \ge a,
\end{cases}
\qquad
\mathbf{E} = \frac{\mathbf{D}}{\varepsilon} .
$$

**Ley de Ampère y teorema de Stokes:**

$$
\nabla\times\mathbf{H} = \mathbf{J}
\quad\Longleftrightarrow\quad
\oint_C \mathbf{H}\cdot d\boldsymbol{\ell} = I_{\text{enc}} .
$$

Para un conductor cilíndrico de radio $R$ con corriente uniforme:

$$
I = J\,\pi R^2,
\qquad
H_\phi(R) = \frac{I}{2\pi R} = \frac{J R}{2}.
$$

**Ley de Ohm puntual:**

$$
\mathbf{J} = \sigma\mathbf{E}
\quad\Longrightarrow\quad
E = \frac{J}{\sigma}.
$$

**Fuerza de Lorentz** sobre una carga $q$ (para un electrón, $q = -e$):

$$
\mathbf{F} = q\left(\mathbf{E} + \mathbf{v}\times\mathbf{B}\right).
$$

## 4. Qué significa físicamente

**El campo máximo está en la superficie.** Dentro crece, fuera decae. Justo en
$r = a$ las dos ramas se encuentran, y ahí está el máximo. Ese empalme sin
saltos no es casualidad: es la condición de borde que estudiaremos en la
semana 4.

**De lejos, la esfera es una carga puntual.** Para $r \gg a$ la fórmula
exterior es idéntica a la de una carga puntual $Q$ en el centro. Lo que pase
adentro deja de importar.

**En el cobre el campo es ridículamente pequeño.** Con $\sigma = 5.8\times
10^{7}$ S/m, sostener unos pocos A/m² requiere un campo del orden de
$10^{-8}$ V/m. Ése es el argumento que justifica tratar los conductores como
equipotenciales.

**La fuerza magnética no hace trabajo.** Como $q\,\mathbf{v}\times\mathbf{B}$
es perpendicular a la velocidad, curva la trayectoria pero no cambia la
rapidez. Y para el electrón hay una inversión extra de signo, porque su carga
es negativa.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: esfera con carga uniforme ---
rho_v = 2.0e-6        # densidad de carga volumétrica [C/m^3]
radio_esfera = 0.08   # radio de la esfera [m]
radios_evaluacion = (0.04, 0.12)  # puntos donde evaluar D y E [m]

# --- Problema 2: conductor, Ohm y Lorentz ---
densidad_corriente = 5.0   # J en el conductor [A/m^2]
radio_conductor = 0.002    # R del conductor [m]
conductividad = 5.8e7      # sigma del cobre [S/m]

velocidad_electron = np.array([2.0e5, 0.0, 0.0])  # v [m/s]
campo_magnetico = np.array([0.0, 0.0, 0.1])       # B [T]

## 6. Implementación

### 6.1 Problema 1 — la esfera

Las funciones vienen de `src/campos_electrostaticos.py`, para que la semana 5
pueda usar exactamente las mismas.

In [ ]:
carga_total = carga_total_esfera_uniforme(radio_esfera, rho_v)

evaluaciones = []
for radio in radios_evaluacion:
    D = float(densidad_flujo_esfera_uniforme(radio, radio_esfera, rho_v))
    E = float(campo_esfera_uniforme(radio, radio_esfera, rho_v))
    region = "interior" if radio < radio_esfera else "exterior"
    evaluaciones.append((radio, region, D, E))

### 6.2 Problema 2 — el conductor y el electrón

In [ ]:
campo_en_conductor = densidad_corriente / conductividad
corriente_total = densidad_corriente * np.pi * radio_conductor**2
campo_H_superficie = corriente_total / (2.0 * np.pi * radio_conductor)

fuerza_lorentz = -CARGA_ELEMENTAL * np.cross(velocidad_electron, campo_magnetico)

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [("Carga total de la esfera", "Q", carga_total, "C")]
    + [
        (f"D en r = {radio:.3f} m ({region})", "D", D, "C/m^2")
        for radio, region, D, _ in evaluaciones
    ]
    + [
        (f"E en r = {radio:.3f} m ({region})", "|E|", E, "V/m")
        for radio, region, _, E in evaluaciones
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Campo eléctrico en el conductor", "E", campo_en_conductor, "V/m"),
        ("Corriente total", "I", corriente_total, "A"),
        ("Campo magnético en r = R", "H_phi(R)", campo_H_superficie, "A/m"),
        ("Fuerza de Lorentz, componente x", "F_x", fuerza_lorentz[0], "N"),
        ("Fuerza de Lorentz, componente y", "F_y", fuerza_lorentz[1], "N"),
        ("Fuerza de Lorentz, componente z", "F_z", fuerza_lorentz[2], "N"),
    ]
)

## 8. Visualización

Cómo varía $|\mathbf{E}|$ con la distancia al centro. La línea vertical marca
la superficie de la esfera.

In [ ]:
r = np.linspace(1.0e-5, 2.0 * radio_esfera, 500)
campo_radial = campo_esfera_uniforme(r, radio_esfera, rho_v)

fig, eje = plt.subplots()
eje.plot(r * 100.0, campo_radial)
eje.axvline(radio_esfera * 100.0, color="black", linestyle="--", label="superficie")
for radio, _, _, E in evaluaciones:
    eje.scatter([radio * 100.0], [E], zorder=5)
eje.set_xlabel("r (cm)")
eje.set_ylabel("|E| (V/m)")
eje.set_title("Campo de una esfera con carga uniforme")
eje.legend()
fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**La curva sube y después baja.** El tramo interior es una recta que sale del
origen: en el centro no hay carga encerrada, así que el campo es cero. El
tramo exterior es una hipérbola. Se juntan sin salto en $r = a$.

**El campo dentro del cobre es despreciable.** Sale del orden de $10^{-8}$
V/m: unas cien mil millones de veces menor que el de la esfera cargada.
Sostener corriente en un buen conductor prácticamente no cuesta campo.

**La fuerza sobre el electrón es transversal.** Con $\mathbf{v}$ en $x$ y
$\mathbf{B}$ en $z$, el producto $\mathbf{v}\times\mathbf{B}$ apunta en $-y$;
al multiplicar por la carga negativa, la fuerza queda en $+y$. La trayectoria
se curva, pero la rapidez no cambia.

## 10. Ejercicios para experimentar

            1. Duplique `rho_v`. ¿Cómo cambian $Q$, $D$ y $E$? ¿Es lineal en las dos
               regiones?
            2. Agregue `radio_esfera` a `radios_evaluacion` para evaluar justo en la
               superficie. Compruebe que las dos ramas dan el mismo valor.
            3. Reduzca `radio_esfera` a `0.04` sin cambiar `rho_v`. ¿El campo máximo sube
               o baja? Use $E_{\max} = \rho_v a/(3\varepsilon_0)$ para explicarlo.
            4. Cambie `conductividad` a `1.0e-4` S/m (agua de mar). ¿Cuánto crece el
               campo interno? ¿Sigue siendo razonable tratarlo como equipotencial?
            5. Invierta el signo de `campo_magnetico`. ¿Qué le pasa a la fuerza?
            6. Ponga `velocidad_electron = [0.0, 0.0, 2.0e5]`, paralela a $\mathbf{B}$.
               Prediga el resultado antes de ejecutar.